In [ ]:
DXF_FOLDER = "/kaggle/input/datasets/cemilfatih/altas-5-profile-cad-files/cad_files"
BG_FOLDER = "/kaggle/input/datasets/cemilfatih/background/background"
BASE_DIR = "dataset_5_profile_FINAL"


In [ ]:
!pip install ezdxf opencv-python numpy

### 1524-1560 profile piles generation

In [ ]:
"""
Script 1/3: 1524 ve 1560 Profilleri (FAST)
Kaynak: V14 (FIXED GEOMETRY) - Paired Block + Void Fill
Optimizasyon: Vectorized NumPy, Sprite Caching, Reduced Depth Steps
"""
import ezdxf
import cv2
import numpy as np
import random
import os
import math
import glob
import networkx as nx

# ==========================================
# ORTAK AYARLAR (3 scriptte de aynı olmalı)
# ==========================================
IMAGES_PER_CLASS_TRAIN = 2000
IMAGES_PER_CLASS_VAL = 50
IMG_WIDTH = 1280
IMG_HEIGHT = 1280


CLASS_ORDER = ['1524', '1560', '9859', '7170', '9794']
CLASS_MAP = {name: i for i, name in enumerate(CLASS_ORDER)}

MY_PROFILES = {
    '1524': os.path.join(DXF_FOLDER, "1524_CLEAN.dxf"),
    '1560': os.path.join(DXF_FOLDER, "1560_CLEAN.dxf"),
}

SPECIAL_PAIRS_CONFIG = {
    '1524': 0.70,
    '1560': 0.85,
}

# ==========================================
# DXF YÜKLEYİCİ (V14 - Multi Contour Graph)
# ==========================================
class DXFProfileLoader:
    def arc_to_points(self, entity, s=30):
        c, r = entity.dxf.center, entity.dxf.radius
        start = math.radians(entity.dxf.start_angle)
        end = math.radians(entity.dxf.end_angle)
        if end < start: end += 2 * math.pi
        return [[c.x + r * math.cos(a), c.y + r * math.sin(a)] for a in np.linspace(start, end, s)]

    def circle_to_points(self, entity, s=60):
        c, r = entity.dxf.center, entity.dxf.radius
        return [[c.x + r * math.cos(a), c.y + r * math.sin(a)] for a in np.linspace(0, 2 * math.pi, s)]

    def __init__(self):
        self.profiles = {}

    def load_dxf(self, dxf_path, profile_name):
        if not os.path.exists(dxf_path):
            print(f"⚠️  {dxf_path} bulunamadı!")
            return None
        try:
            doc = ezdxf.readfile(dxf_path)
            msp = doc.modelspace()
            G = nx.Graph()
            def to_key(p): return (round(p[0], 3), round(p[1], 3))
            has_data = False
            for e in msp:
                if e.dxftype() == 'LINE':
                    G.add_edge(to_key(e.dxf.start), to_key(e.dxf.end)); has_data = True
                elif e.dxftype() == 'LWPOLYLINE':
                    pts = e.get_points('xy')
                    for i in range(len(pts)-1): G.add_edge(to_key(pts[i]), to_key(pts[i+1]))
                    if e.closed: G.add_edge(to_key(pts[-1]), to_key(pts[0]))
                    has_data = True
                elif e.dxftype() == 'ARC':
                    pts = self.arc_to_points(e)
                    for i in range(len(pts)-1): G.add_edge(to_key(pts[i]), to_key(pts[i+1]))
                    has_data = True
                elif e.dxftype() == 'CIRCLE':
                    pts = self.circle_to_points(e)
                    for i in range(len(pts)-1): G.add_edge(to_key(pts[i]), to_key(pts[i+1]))
                    G.add_edge(to_key(pts[-1]), to_key(pts[0]))
                    has_data = True
            if not has_data: return None
            components = list(nx.connected_components(G))
            main_comp = max(components, key=len)
            main_pts = np.array(list(main_comp))
            main_min, main_max = np.min(main_pts, axis=0), np.max(main_pts, axis=0)
            valid_comps = []
            for comp in components:
                if len(comp) < 3: continue
                pts = np.array(list(comp))
                c_min, c_max = np.min(pts, axis=0), np.max(pts, axis=0)
                if comp == main_comp or (np.all(c_min >= main_min - 0.1) and np.all(c_max <= main_max + 0.1)):
                    valid_comps.append(comp)
            if not valid_comps: return None
            contours = []
            all_pts = []
            for comp in valid_comps:
                subgraph = G.subgraph(comp)
                start_node = next((n for n, d in subgraph.degree() if d == 1), list(comp)[0])
                ordered_nodes = list(nx.dfs_preorder_nodes(subgraph, source=start_node))
                contours.append(np.array(ordered_nodes))
                all_pts.extend(ordered_nodes)
            all_pts = np.array(all_pts)
            min_vals, max_vals = np.min(all_pts, axis=0), np.max(all_pts, axis=0)
            center = (min_vals + max_vals) / 2
            max_dim = np.max(max_vals - min_vals)
            normalized_contours = []
            for pts in contours:
                pts = pts - center
                if max_dim > 0: pts = pts / max_dim
                normalized_contours.append(pts)
            self.profiles[profile_name] = normalized_contours
            print(f"✅ Yüklendi: {profile_name} ({len(normalized_contours)} contour)")
            return normalized_contours
        except Exception as e:
            print(f"❌ DXF Hatası ({profile_name}): {e}")
            return None

# ==========================================
# RENDER MOTORU (V14 - FAST VECTORIZED)
# ==========================================
class ProfileRenderer:
    def __init__(self): pass

    def apply_aluminum_texture(self, mask):
        h, w = mask.shape
        base_val = random.randint(180, 220)
        # FAST: vectorized gradient
        if random.random() < 0.5:
            gradient = np.clip(base_val - 25 * np.sin(np.arange(h, dtype=np.float32) / h * 3.14), 0, 255).astype(np.uint8)
            img_bgr = np.tile(gradient[:, np.newaxis, np.newaxis], (1, w, 3))
        else:
            gradient = np.clip(base_val - 25 * np.sin(np.arange(w, dtype=np.float32) / w * 3.14), 0, 255).astype(np.uint8)
            img_bgr = np.tile(gradient[np.newaxis, :, np.newaxis], (h, 1, 3))
        noise = np.random.randint(-15, 15, img_bgr.shape, dtype=np.int16)
        img_bgr = np.clip(img_bgr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        img_bgr = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
        contours_cv, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_bgr, contours_cv, -1, (60, 60, 60), 1, cv2.LINE_AA)
        return cv2.merge((img_bgr, mask))

    def get_single_half(self, contours, size, rotation=0):
        canvas_size = int(size * 2.5)
        pts_list = []
        for contour in contours:
            scaled = contour * size + canvas_size // 2
            if rotation != 0:
                M = cv2.getRotationMatrix2D((canvas_size//2, canvas_size//2), rotation, 1.0)
                ones = np.ones((len(scaled), 1))
                scaled = M.dot(np.hstack([scaled, ones]).T).T
            pts_list.append(scaled.astype(np.int32))
        mask = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        cv2.fillPoly(mask, pts_list, 255)
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        coords = cv2.findNonZero(mask)
        if coords is None: return None, 0
        x, y, w, h = cv2.boundingRect(coords)
        cropped_mask = mask[y:y+h, x:x+w]
        thickness = int(w * 0.12)
        return self.apply_aluminum_texture(cropped_mask), thickness

    def create_paired_block(self, contours, size, profile_name, target_rotation=0):
        sprite_a, t = self.get_single_half(contours, size, 0)
        if sprite_a is None: return np.zeros((10,10,4), dtype=np.uint8), (0,0,0,0)
        h, w = sprite_a.shape[:2]
        sprite_b_raw, _ = self.get_single_half(contours, size, 180)
        sprite_b = cv2.resize(sprite_b_raw, (w, h))
        tightness = SPECIAL_PAIRS_CONFIG.get(profile_name, 1.0)
        shift_val = int(t * tightness)
        canvas_w = w * 2 + shift_val * 4
        canvas_h = h * 2 + shift_val * 4
        canvas = np.zeros((canvas_h, canvas_w, 4), dtype=np.uint8)
        cx, cy = canvas_w // 2, canvas_h // 2
        pos_a_x, pos_a_y = cx - w // 2, cy - h // 2
        pos_b_x, pos_b_y = pos_a_x + shift_val, pos_a_y + shift_val

        # VOID FILLING - FULLY VECTORIZED
        void_x1 = min(pos_a_x, pos_b_x) + t
        void_y1 = min(pos_a_y, pos_b_y) + t
        void_x2 = max(pos_a_x + w, pos_b_x + w) - t
        void_y2 = max(pos_a_y + h, pos_b_y + h) - t
        if void_x2 > void_x1 and void_y2 > void_y1:
            vw, vh = void_x2 - void_x1, void_y2 - void_y1
            if profile_name == '1560':
                base_val = np.random.randint(50, 90, (vh, vw), dtype=np.uint8)
            else:
                base_val = np.random.randint(10, 30, (vh, vw), dtype=np.uint8)
            void_img_bgr = np.stack([base_val]*3, axis=-1)
            noise = np.random.randint(-20, 20, (vh, vw, 3), dtype=np.int16)
            void_img_bgr = np.clip(void_img_bgr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
            # Vectorized vignette
            Y, X = np.ogrid[:vh, :vw]
            dist_map = np.sqrt(((X - vw//2)**2 + (Y - vh//2)**2).astype(np.float32))
            max_dist = math.sqrt((vw/2)**2 + (vh/2)**2)
            if max_dist > 0:
                factor = np.clip(1.0 - (dist_map / max_dist) * 0.8, 0, 1)
                void_img_bgr = (void_img_bgr.astype(np.float32) * factor[:,:,np.newaxis]).astype(np.uint8)
            void_alpha = np.full((vh, vw), 255, dtype=np.uint8)
            void_img = np.dstack([void_img_bgr, void_alpha])
            canvas[void_y1:void_y1+vh, void_x1:void_x1+vw] = void_img

        def paste(bg, fg, x, y):
            fh, fw = fg.shape[:2]
            if x+fw > bg.shape[1] or y+fh > bg.shape[0]: return bg
            alpha = fg[:,:,3:4].astype(np.float32) / 255.0
            bg[y:y+fh, x:x+fw, :3] = (fg[:,:,:3].astype(np.float32) * alpha + bg[y:y+fh, x:x+fw, :3].astype(np.float32) * (1-alpha)).astype(np.uint8)
            bg[y:y+fh, x:x+fw, 3] = np.maximum(bg[y:y+fh, x:x+fw, 3], fg[:,:,3])
            return bg
        canvas = paste(canvas, sprite_a, pos_a_x, pos_a_y)
        canvas = paste(canvas, sprite_b, pos_b_x, pos_b_y)
        if target_rotation in (90, 270):
            canvas = cv2.rotate(canvas, cv2.ROTATE_90_CLOCKWISE)
        coords = cv2.findNonZero(canvas[:,:,3])
        if coords is not None:
            x, y, w_crop, h_crop = cv2.boundingRect(coords)
            return canvas[y:y+h_crop, x:x+w_crop], (x, y, w_crop, h_crop)
        return canvas, (0,0,0,0)

    def render_profile_sprite(self, contours, size, rotation=0, profile_name="default"):
        if profile_name in SPECIAL_PAIRS_CONFIG:
            return self.create_paired_block(contours, size, profile_name, target_rotation=rotation)
        sprite, t = self.get_single_half(contours, size, rotation)
        if sprite is None: return np.zeros((10,10,4), dtype=np.uint8), (0,0,0,0)
        return sprite, (0,0,sprite.shape[1], sprite.shape[0])

# ==========================================
# PALET ÜRETİCİ (V14 - FAST)
# ==========================================
class SyntheticPalletGenerator:
    def __init__(self, loader, renderer, bg_folder='background'):
        self.loader = loader
        self.renderer = renderer
        self.width = IMG_WIDTH
        self.height = IMG_HEIGHT
        self.bg_images = glob.glob(os.path.join(bg_folder, "*.*"))

    def get_random_background(self):
        if self.bg_images and random.random() < 0.95:
            bg_path = random.choice(self.bg_images)
            bg = cv2.imread(bg_path)
            if bg is not None:
                bg = cv2.resize(bg, (self.width, self.height))
                return (bg.astype(np.float32) * random.uniform(0.5, 0.9)).astype(np.uint8)
        val = random.randint(40, 100)
        bg = np.full((self.height, self.width, 3), val, dtype=np.uint8)
        noise = np.random.randint(-30, 30, bg.shape, dtype=np.int16)
        return np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    def overlay_fast(self, img, overlay, pos_x, pos_y, alpha):
        h_c, w_c = img.shape[:2]
        oh, ow = overlay.shape[:2]
        px, py = int(pos_x), int(pos_y)
        x1, y1 = max(0, px), max(0, py)
        x2, y2 = min(w_c, px+ow), min(h_c, py+oh)
        if x2 <= x1 or y2 <= y1: return
        ox1, oy1 = x1-px, y1-py
        hs, ws = y2-y1, x2-x1
        a = alpha[oy1:oy1+hs, ox1:ox1+ws].astype(np.float32) / 255.0
        a3 = a[:,:,np.newaxis]
        roi = img[y1:y2, x1:x2]
        fg = overlay[oy1:oy1+hs, ox1:ox1+ws, :3]
        img[y1:y2, x1:x2] = (fg.astype(np.float32)*a3 + roi.astype(np.float32)*(1-a3)).astype(np.uint8)

    def check_collision(self, rect, rects):
        nx_r, ny_r, nw, nh = rect
        for ox, oy, ow, oh in rects:
            if nx_r < ox+ow and nx_r+nw > ox and ny_r < oy+oh and ny_r+nh > oy: return True
        return False

    def generate_pallet(self, ptype):
        pallet = self.get_random_background()
        anns = []
        contour = self.loader.profiles[ptype]
        sf = self.width / 640.0
        depth_cmds, face_cmds = [], []
        occupied = []

        for _ in range(random.randint(1, 2)):
            rot = 0
            bsz = int(random.randint(60, 70) * sf)
            if ptype == '1560':
                if random.random() < 0.5:
                    rot = 90; cols, rows = random.randint(1,2), random.randint(8,14)
                else:
                    cols, rows = random.randint(5,12), random.randint(1,4)
            else:
                if random.random() < 0.3: cols, rows = 1, random.randint(5,12)
                else: cols, rows = random.randint(3,6), random.randint(3,8)

            # CACHE: 1 kez üret
            cached, _ = self.renderer.render_profile_sprite(contour, bsz, rot, ptype)
            ph, pw = cached.shape[:2]
            if ph == 0 or pw == 0: continue
            pw_total, ph_total = cols*pw, rows*ph
            mx, my = max(0, self.width-pw_total), max(0, self.height-ph_total)
            found = False
            for _ in range(50):
                sx, sy = random.randint(0,mx), random.randint(0,my)
                r = [sx, sy, pw_total, ph_total]
                if not self.check_collision(r, occupied):
                    occupied.append(r); found = True; break
            if not found: continue

            cached_alpha = cached[:,:,3]
            for ri in range(rows):
                cy = sy + ri*ph
                for ci in range(cols):
                    cx = sx + ci*pw
                    if cx+pw > self.width or cy+ph > self.height: continue
                    fx, fy = cx+random.randint(-1,1), cy+random.randint(-1,1)
                    ocx, ocy = fx+pw//2, fy+ph//2
                    vx, vy = self.width//2-ocx, self.height//2-ocy
                    d = math.sqrt(vx**2+vy**2)
                    md = math.sqrt((self.width/2)**2+(self.height/2)**2)
                    dl = min((d/md)*(50*sf), pw*0.8)
                    depth_cmds.append({'x':fx,'y':fy,'sw':pw,'sh':ph,'vx':vx,'vy':vy,'d':d,'dl':dl,'alpha':cached_alpha})
                    face_cmds.append({'sprite':cached,'x':fx,'y':fy,'sw':pw,'sh':ph})

        # DEPTH (reduced steps, step=2)
        for c in depth_cmds:
            if c['d'] <= 0: continue
            nx_v, ny_v = c['vx']/c['d'], c['vy']/c['d']
            tox, toy = int(nx_v*c['dl']), int(ny_v*c['dl'])
            body = np.full((c['sh'], c['sw'], 3), 50, dtype=np.uint8)
            steps = min(int(max(abs(tox), abs(toy))), 15)
            for s in range(steps, 0, -2):
                r = s/steps if steps > 0 else 0
                self.overlay_fast(pallet, body, c['x']+int(tox*r), c['y']+int(toy*r), c['alpha'])

        cls_id = CLASS_MAP[ptype]
        for c in face_cmds:
            self.overlay_fast(pallet, c['sprite'], c['x'], c['y'], c['sprite'][:,:,3])
            anns.append({'class_id': cls_id, 'bbox': [
                (c['x']+c['sw']/2)/self.width, (c['y']+c['sh']/2)/self.height,
                c['sw']/self.width, c['sh']/self.height]})
        return pallet, anns

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)

def main():
    loader = DXFProfileLoader()
    for name, path in MY_PROFILES.items(): loader.load_dxf(path, name)
    if not loader.profiles: print("❌ Profil yüklenemedi!"); return
    renderer = ProfileRenderer()
    gen = SyntheticPalletGenerator(loader, renderer, BG_FOLDER)
    for s in ['train','val']:
        os.makedirs(f'{BASE_DIR}/images/{s}', exist_ok=True)
        os.makedirs(f'{BASE_DIR}/labels/{s}', exist_ok=True)
    print(f"\n🚀 Script 1/3 FAST: 1524 + 1560 Başlıyor...")
    for ptype in MY_PROFILES:
        if ptype not in loader.profiles: continue
        print(f"\n📦 {ptype} ...")
        for i in range(IMAGES_PER_CLASS_TRAIN):
            if i % 20 == 0: print(f"   [Train] {i}/{IMAGES_PER_CLASS_TRAIN}")
            p, a = gen.generate_pallet(ptype)
            n = f"{ptype}_train_{i}"
            cv2.imwrite(f'{BASE_DIR}/images/train/{n}.jpg', p)
            #cv2.imwrite(f'{BASE_DIR}/images/train/{n}_clahe.jpg', apply_clahe(p))
            ls = "".join([f"{x['class_id']} {x['bbox'][0]:.6f} {x['bbox'][1]:.6f} {x['bbox'][2]:.6f} {x['bbox'][3]:.6f}\n" for x in a])
            #for suffix in [f'{n}.txt', f'{n}_clahe.txt']:
            for suffix in [f'{n}.txt']:
                with open(f'{BASE_DIR}/labels/train/{suffix}', 'w') as f: f.write(ls)
        for i in range(IMAGES_PER_CLASS_VAL):
            if i % 5 == 0: print(f"   [Val] {i}/{IMAGES_PER_CLASS_VAL}")
            p, a = gen.generate_pallet(ptype)
            n = f"{ptype}_val_{i}"
            cv2.imwrite(f'{BASE_DIR}/images/val/{n}.jpg', p)
            ls = "".join([f"{x['class_id']} {x['bbox'][0]:.6f} {x['bbox'][1]:.6f} {x['bbox'][2]:.6f} {x['bbox'][3]:.6f}\n" for x in a])
            with open(f'{BASE_DIR}/labels/val/{n}.txt', 'w') as f: f.write(ls)
    print(f"\n✅ Script 1/3 TAMAMLANDI!")

if __name__ == "__main__":
    main()


### 7170-9794 profile piles generation

In [ ]:
"""
Script 3/3: 7170 ve 9794 Profilleri (FAST)
Kaynak: V24 (PITCH BLACK TUNNEL) - Scattered + Tunnel Shadow
Optimizasyon: Vectorized texture, reduced depth steps, fast overlay
"""
import ezdxf
import cv2
import numpy as np
import random
import os
import math
import glob
import networkx as nx

# ==========================================
# ORTAK AYARLAR (3 scriptte de aynı olmalı)
# ==========================================
IMAGES_PER_CLASS_TRAIN = 2000
IMAGES_PER_CLASS_VAL = 50
IMG_WIDTH = 1280
IMG_HEIGHT = 1280


CLASS_ORDER = ['1524', '1560', '9859', '7170', '9794']
CLASS_MAP = {name: i for i, name in enumerate(CLASS_ORDER)}

MY_PROFILES = {
    '7170': os.path.join(DXF_FOLDER, "7170.dxf"),
    '9794': os.path.join(DXF_FOLDER, "9794.dxf"),
}

# ==========================================
# DXF YÜKLEYİCİ (V24 - Single Contour)
# ==========================================
class DXFProfileLoader:
    def __init__(self):
        self.profiles = {}

    def load_dxf(self, dxf_path, profile_name):
        if not os.path.exists(dxf_path):
            print(f"⚠️  {dxf_path} bulunamadı!")
            return None
        try:
            doc = ezdxf.readfile(dxf_path)
            msp = doc.modelspace()
            G = nx.Graph()
            def to_key(p): return (round(p[0], 3), round(p[1], 3))
            has_data = False
            for e in msp:
                if e.dxftype() == 'LINE':
                    G.add_edge(to_key(e.dxf.start), to_key(e.dxf.end)); has_data = True
                elif e.dxftype() == 'LWPOLYLINE':
                    pts = e.get_points('xy')
                    for i in range(len(pts)-1): G.add_edge(to_key(pts[i]), to_key(pts[i+1]))
                    if e.closed: G.add_edge(to_key(pts[-1]), to_key(pts[0]))
                    has_data = True
            if not has_data: return None
            components = list(nx.connected_components(G))
            largest = max(components, key=len)
            subgraph = G.subgraph(largest)
            ordered = list(nx.dfs_preorder_nodes(subgraph))
            pts = np.array(ordered)
            min_v, max_v = np.min(pts, axis=0), np.max(pts, axis=0)
            center = (min_v + max_v) / 2
            pts = pts - center
            h = max_v[1] - min_v[1]
            self.profiles[profile_name] = pts / h if h > 0 else pts
            print(f"✅ Yüklendi: {profile_name}")
            return self.profiles[profile_name]
        except Exception as e:
            print(f"❌ DXF Hatası ({profile_name}): {e}")
            return None

# ==========================================
# RENDER MOTORU (V24 - FAST)
# ==========================================
class ProfileRenderer:
    def __init__(self): pass

    def apply_aluminum_texture(self, mask):
        h, w = mask.shape
        is_shiny = random.random() < 0.30
        if is_shiny:
            base_val, contrast = random.randint(230, 255), 40
        else:
            base_val, contrast = random.randint(180, 220), 25
        # FAST: vectorized
        if random.random() < 0.5:
            grad = np.clip(base_val - contrast * np.sin(np.arange(h, dtype=np.float32) / h * 3.14), 0, 255).astype(np.uint8)
            img_bgr = np.tile(grad[:, np.newaxis, np.newaxis], (1, w, 3))
        else:
            grad = np.clip(base_val - contrast * np.sin(np.arange(w, dtype=np.float32) / w * 3.14), 0, 255).astype(np.uint8)
            img_bgr = np.tile(grad[np.newaxis, :, np.newaxis], (h, 1, 3))
        noise = np.random.randint(-15, 15, img_bgr.shape, dtype=np.int16)
        img_bgr = np.clip(img_bgr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        img_bgr = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
        contours_cv, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_bgr, contours_cv, -1, (60, 60, 60), 1, cv2.LINE_AA)
        return cv2.merge((img_bgr, mask))

    def get_single_half(self, contour, size, rotation=0):
        canvas_size = int(size * 2.5)
        scaled = contour * size + canvas_size // 2
        if rotation != 0:
            M = cv2.getRotationMatrix2D((canvas_size//2, canvas_size//2), rotation, 1.0)
            ones = np.ones((len(scaled), 1))
            scaled = M.dot(np.hstack([scaled, ones]).T).T
        pts = scaled.astype(np.int32)
        mask = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        cv2.fillPoly(mask, [pts], 255)
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        coords = cv2.findNonZero(mask)
        if coords is None: return None, 0
        x, y, w, h = cv2.boundingRect(coords)
        thickness = int(min(w, h) * 0.15)
        return self.apply_aluminum_texture(mask[y:y+h, x:x+w]), thickness

# ==========================================
# PALET ÜRETİCİ (V24 - FAST)
# ==========================================
class SyntheticPalletGenerator:
    def __init__(self, loader, renderer, bg_folder='background'):
        self.loader = loader
        self.renderer = renderer
        self.width = IMG_WIDTH
        self.height = IMG_HEIGHT
        self.bg_images = glob.glob(os.path.join(bg_folder, "*.*"))

    def get_random_background(self):
        if self.bg_images and random.random() < 0.95:
            bg = cv2.imread(random.choice(self.bg_images))
            if bg is not None:
                bg = cv2.resize(bg, (self.width, self.height))
                return (bg.astype(np.float32) * random.uniform(0.5, 0.9)).astype(np.uint8)
        val = random.randint(40, 100)
        bg = np.full((self.height, self.width, 3), val, dtype=np.uint8)
        noise = np.random.randint(-30, 30, bg.shape, dtype=np.int16)
        return np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    def generate_tunnel_void(self, w, h, ocx, ocy, thickness):
        vw, vh = w - 2*thickness, h - 2*thickness
        if vw <= 0 or vh <= 0: return None
        vx, vy = self.width//2 - ocx, self.height//2 - ocy
        lcx = np.clip(vw//2 + int(vx*0.3), 0, vw)
        lcy = np.clip(vh//2 + int(vy*0.3), 0, vh)
        Y, X = np.ogrid[:vh, :vw]
        dm = np.sqrt(((X-lcx)**2 + (Y-lcy)**2).astype(np.float32))
        md = np.sqrt(float(vw**2+vh**2))
        nd = np.clip(dm / (md*0.6), 0, 1)
        opacity = random.uniform(0.80, 0.95)
        alpha = (np.power(nd, 0.7) * opacity * 255).astype(np.uint8)
        base = np.random.randint(0, 20, (vh, vw, 3), dtype=np.uint8)
        return np.dstack([base, alpha])

    def overlay_fast(self, img, overlay, px, py, alpha):
        hc, wc = img.shape[:2]
        oh, ow = overlay.shape[:2]
        px, py = int(px), int(py)
        x1, y1 = max(0,px), max(0,py)
        x2, y2 = min(wc,px+ow), min(hc,py+oh)
        if x2<=x1 or y2<=y1: return
        ox, oy = x1-px, y1-py
        hs, ws = y2-y1, x2-x1
        a = alpha[oy:oy+hs, ox:ox+ws].astype(np.float32)/255.0
        a3 = a[:,:,np.newaxis]
        fg = overlay[oy:oy+hs, ox:ox+ws, :3].astype(np.float32)
        bg = img[y1:y2, x1:x2].astype(np.float32)
        img[y1:y2, x1:x2] = (fg*a3 + bg*(1-a3)).astype(np.uint8)

    def generate_pallet(self, ptype):
        pallet = self.get_random_background()
        anns = []
        sf = self.width / 640.0
        contour = self.loader.profiles[ptype]
        bsz = int(random.randint(25, 40) * sf)
        spacer_r = (int(5*sf), int(15*sf))
        gap_r = (int(2*sf), int(5*sf))

        ref, ref_t = self.renderer.get_single_half(contour, bsz, 0)
        if ref is None: return pallet, []
        ph, pw = ref.shape[:2]
        if ph == 0 or pw == 0: return pallet, []

        cur_y = self.height - int(10*sf)
        depth_cmds, face_cmds = [], []

        while cur_y > int(50*sf):
            spacer = random.randint(*spacer_r)
            row_y = cur_y - spacer - ph
            if row_y < 0: break
            cur_x = random.randint(0, int(30*sf))
            while cur_x < self.width - pw:
                rot = random.choice([0, 180])
                sz = int(bsz * random.uniform(0.95, 1.05))
                sprite, t = self.renderer.get_single_half(contour, sz, rot)
                if sprite is None: break
                sh, sw = sprite.shape[:2]
                px, py = cur_x, row_y + random.randint(-2, 2)
                if px + sw > self.width: break
                ocx, ocy = px+sw//2, py+sh//2
                vx, vy = self.width//2 - ocx, self.height//2 - ocy
                d = math.sqrt(vx**2+vy**2)
                md = math.sqrt((self.width/2)**2+(self.height/2)**2)
                dl = min((d/md)*(50*sf), sw*0.8)
                # Tunnel void (V24)
                void = self.generate_tunnel_void(sw, sh, ocx, ocy, t)
                if void is not None:
                    self.overlay_fast(pallet, void, px+t, py+t, void[:,:,3])
                depth_cmds.append({'x':px,'y':py,'sw':sw,'sh':sh,'vx':vx,'vy':vy,'d':d,'dl':dl,'alpha':sprite[:,:,3]})
                face_cmds.append({'sprite':sprite,'x':px,'y':py,'sw':sw,'sh':sh})
                cur_x += sw + random.randint(*gap_r)
            cur_y = row_y

        # Depth (reduced: max 15 steps, step=2)
        for c in depth_cmds:
            if c['d'] <= 0: continue
            nx_v, ny_v = c['vx']/c['d'], c['vy']/c['d']
            tox, toy = int(nx_v*c['dl']), int(ny_v*c['dl'])
            body = np.full((c['sh'], c['sw'], 3), 50, dtype=np.uint8)
            steps = min(int(max(abs(tox), abs(toy))), 15)
            for s in range(steps, 0, -2):
                r = s/steps if steps>0 else 0
                self.overlay_fast(pallet, body, c['x']+int(tox*r), c['y']+int(toy*r), c['alpha'])

        cls_id = CLASS_MAP[ptype]
        for c in face_cmds:
            self.overlay_fast(pallet, c['sprite'], c['x'], c['y'], c['sprite'][:,:,3])
            anns.append({'class_id': cls_id, 'bbox': [
                (c['x']+c['sw']/2)/self.width, (c['y']+c['sh']/2)/self.height,
                c['sw']/self.width, c['sh']/self.height]})

        if random.random() < 0.3:
            noise = np.random.normal(0, 3, pallet.shape).astype(np.int16)
            pallet = np.clip(pallet.astype(np.int16)+noise, 0, 255).astype(np.uint8)
        return pallet, anns

def main():
    loader = DXFProfileLoader()
    for name, path in MY_PROFILES.items(): loader.load_dxf(path, name)
    if not loader.profiles: print("❌ Profil yüklenemedi!"); return
    renderer = ProfileRenderer()
    gen = SyntheticPalletGenerator(loader, renderer, BG_FOLDER)
    for s in ['train','val']:
        os.makedirs(f'{BASE_DIR}/images/{s}', exist_ok=True)
        os.makedirs(f'{BASE_DIR}/labels/{s}', exist_ok=True)
    print(f"\n🚀 Script 3/3 FAST: 7170 + 9794 Başlıyor...")
    for ptype in MY_PROFILES:
        if ptype not in loader.profiles: continue
        print(f"\n📦 {ptype} ...")
        for i in range(IMAGES_PER_CLASS_TRAIN):
            if i % 20 == 0: print(f"   [Train] {i}/{IMAGES_PER_CLASS_TRAIN}")
            p, a = gen.generate_pallet(ptype)
            n = f"{ptype}_train_{i}"
            cv2.imwrite(f'{BASE_DIR}/images/train/{n}.jpg', p)
            ls = "".join([f"{x['class_id']} {x['bbox'][0]:.6f} {x['bbox'][1]:.6f} {x['bbox'][2]:.6f} {x['bbox'][3]:.6f}\n" for x in a])
            with open(f'{BASE_DIR}/labels/train/{n}.txt', 'w') as f: f.write(ls)
        for i in range(IMAGES_PER_CLASS_VAL):
            if i % 5 == 0: print(f"   [Val] {i}/{IMAGES_PER_CLASS_VAL}")
            p, a = gen.generate_pallet(ptype)
            n = f"{ptype}_val_{i}"
            cv2.imwrite(f'{BASE_DIR}/images/val/{n}.jpg', p)
            ls = "".join([f"{x['class_id']} {x['bbox'][0]:.6f} {x['bbox'][1]:.6f} {x['bbox'][2]:.6f} {x['bbox'][3]:.6f}\n" for x in a])
            with open(f'{BASE_DIR}/labels/val/{n}.txt', 'w') as f: f.write(ls)
    print(f"\n✅ Script 3/3 TAMAMLANDI!")

if __name__ == "__main__":
    main()


### 9859 profile piles generation

In [ ]:
"""
Script 2/3: 9859 Profili (TURBO)
Kaynak: Test Tuner - Package + Pattern + Tunnel Void

HIZLANDIRMA:
1. DXF parse + mask + texture = 1 KEZ (başta)
2. Paket sprite = 1 KEZ (başta) 
3. Flipped paket = 1 KEZ (başta)
4. Tunnel void = boyut bazlı cache (aynı boyut tekrar hesaplanmaz)
5. Canvas boyutu kontrollü (gereksiz büyük canvas yok)
6. Depth step azaltıldı + step=2
7. Texture vectorized
"""
import ezdxf
import cv2
import numpy as np
import os
import random
import math
import glob

# ==========================================
# ORTAK AYARLAR (3 scriptte de aynı olmalı)
# ==========================================
IMAGES_PER_CLASS_TRAIN = 2000
IMAGES_PER_CLASS_VAL = 50
IMG_WIDTH = 1280
IMG_HEIGHT = 1280


CLASS_ORDER = ['1524', '1560', '9859', '7170', '9794']
CLASS_MAP = {name: i for i, name in enumerate(CLASS_ORDER)}

DXF_PROFILE = os.path.join(DXF_FOLDER, "9859.dxf")
RENDER_SIZE = 150
PROFILES_PER_PKG = 6

# ==========================================
# DXF -> MASK (1 kez çalışır)
# ==========================================
def extract_mask_from_dxf(dxf_path, canvas_size):
    """DXF dosyasını okur, XOR polygon mask döndürür."""
    if not os.path.exists(dxf_path):
        return None
    doc = ezdxf.readfile(dxf_path)
    msp = doc.modelspace()
    lines = []
    for e in msp:
        if e.dxftype() == 'LINE':
            lines.append(((e.dxf.start.x, e.dxf.start.y), (e.dxf.end.x, e.dxf.end.y)))
        elif e.dxftype() == 'LWPOLYLINE':
            pts = e.get_points('xy')
            for i in range(len(pts)-1):
                lines.append((pts[i], pts[i+1]))
            if e.closed:
                lines.append((pts[-1], pts[0]))
        elif e.dxftype() == 'ARC':
            c, r = e.dxf.center, e.dxf.radius
            sa, ea = math.radians(e.dxf.start_angle), math.radians(e.dxf.end_angle)
            if ea < sa: ea += 2 * math.pi
            pts = [[c.x + r * math.cos(a), c.y + r * math.sin(a)] for a in np.linspace(sa, ea, 30)]
            for i in range(len(pts)-1):
                lines.append((pts[i], pts[i+1]))
        elif e.dxftype() == 'CIRCLE':
            c, r = e.dxf.center, e.dxf.radius
            pts = [[c.x + r * math.cos(a), c.y + r * math.sin(a)] for a in np.linspace(0, 2 * math.pi, 60)]
            for i in range(len(pts)-1):
                lines.append((pts[i], pts[i+1]))
    if not lines:
        return None

    all_pts = np.array([pt for p1, p2 in lines for pt in (p1, p2)])
    min_vals, max_vals = np.min(all_pts, axis=0), np.max(all_pts, axis=0)
    max_dim = max(max_vals[0]-min_vals[0], max_vals[1]-min_vals[1])
    if max_dim == 0: max_dim = 1
    padding = int(canvas_size * 0.1)
    scale = (canvas_size - 2*padding) / max_dim

    scaled_lines = []
    for p1, p2 in lines:
        x1 = (p1[0]-min_vals[0])*scale + padding
        y1 = canvas_size - ((p1[1]-min_vals[1])*scale + padding)
        x2 = (p2[0]-min_vals[0])*scale + padding
        y2 = canvas_size - ((p2[1]-min_vals[1])*scale + padding)
        scaled_lines.append(((x1, y1), (x2, y2)))

    # Edge chaining -> polygon
    polygons = get_polygons_from_lines(scaled_lines, tolerance=3.0)
    polygons = sorted(polygons, key=cv2.contourArea, reverse=True)
    mask = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
    for poly in polygons:
        temp = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        cv2.fillPoly(temp, [poly], 255)
        mask = cv2.bitwise_xor(mask, temp)
    return mask


def get_polygons_from_lines(lines, tolerance=3.0):
    edges = [[np.array(p1), np.array(p2)] for p1, p2 in lines]
    polygons = []
    while edges:
        current_poly = [edges[0][0], edges[0][1]]
        edges.pop(0)
        while True:
            last_pt = current_poly[-1]
            best_dist, best_idx, best_reverse = float('inf'), -1, False
            for i, edge in enumerate(edges):
                d1 = np.linalg.norm(edge[0] - last_pt)
                d2 = np.linalg.norm(edge[1] - last_pt)
                if d1 < best_dist:
                    best_dist, best_idx, best_reverse = d1, i, False
                if d2 < best_dist:
                    best_dist, best_idx, best_reverse = d2, i, True
            if best_idx != -1 and best_dist <= tolerance:
                edge = edges.pop(best_idx)
                current_poly.append(edge[0] if best_reverse else edge[1])
            else:
                break
        if len(current_poly) > 2:
            polygons.append(np.array(current_poly, dtype=np.int32))
    return polygons


def apply_aluminum_texture(mask):
    """Vectorized alüminyum dokusu."""
    h, w = mask.shape
    is_shiny = random.random() < 0.3
    base_val = random.randint(200, 240) if is_shiny else random.randint(150, 190)
    contrast = 30 if is_shiny else 15
    if random.random() < 0.5:
        grad = np.clip(base_val - contrast * np.sin(np.arange(h, dtype=np.float32) / h * 6.28), 0, 255).astype(np.uint8)
        img_bgr = np.tile(grad[:, np.newaxis, np.newaxis], (1, w, 3))
    else:
        grad = np.clip(base_val - contrast * np.sin(np.arange(w, dtype=np.float32) / w * 6.28), 0, 255).astype(np.uint8)
        img_bgr = np.tile(grad[np.newaxis, :, np.newaxis], (h, 1, 3))
    noise = np.random.randint(-12, 12, img_bgr.shape, dtype=np.int16)
    img_bgr = np.clip(img_bgr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    img_bgr = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
    contours_cv, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img_bgr, contours_cv, -1, (50, 50, 50), 1, cv2.LINE_AA)
    return cv2.merge((img_bgr, mask))


def build_base_sprite(dxf_path, size):
    """DXF'den tek profil sprite'ı üretir. 1 KEZ çağrılır."""
    canvas_size = int(size * 3.0)
    mask = extract_mask_from_dxf(dxf_path, canvas_size)
    if mask is None:
        return None
    mask = cv2.GaussianBlur(mask, (3, 3), 0)
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(mask)
    if coords is None:
        return None
    x, y, w, h = cv2.boundingRect(coords)
    textured = apply_aluminum_texture(mask[y:y+h, x:x+w])
    # Yataysa dik çevir
    if textured.shape[1] > textured.shape[0]:
        textured = cv2.rotate(textured, cv2.ROTATE_90_CLOCKWISE)
    return textured


def build_package(sprite, n=6):
    """6 profili yan yana birleştirip paket yapar. 1 KEZ çağrılır."""
    h, w = sprite.shape[:2]
    pkg = np.zeros((h, w * n, 4), dtype=np.uint8)
    for i in range(n):
        pkg[:, i*w:(i+1)*w] = sprite
    return pkg


# ==========================================
# PALET ÜRETİCİ
# ==========================================
class PalletGenerator:
    def __init__(self, pkg_normal, pkg_flipped, pkg_rotated, bg_folder):
        """
        Önceden üretilmiş 3 paket varyantını alır:
        - pkg_normal: düz paket
        - pkg_flipped: 180° döndürülmüş (alternating satırlar için)
        - pkg_rotated: 90° döndürülmüş (vertical satırlar için)
        """
        self.pkg_normal = pkg_normal
        self.pkg_flipped = pkg_flipped
        self.pkg_rotated = pkg_rotated
        self.pkg_h, self.pkg_w = pkg_normal.shape[:2]
        self.rot_h, self.rot_w = pkg_rotated.shape[:2]
        self.width = IMG_WIDTH
        self.height = IMG_HEIGHT
        self.bg_images = glob.glob(os.path.join(bg_folder, "*.*"))
        # Tunnel void cache: (w,h) -> void template
        self._void_cache = {}

    def overlay_fast(self, img, overlay, px, py, alpha):
        hc, wc = img.shape[:2]
        oh, ow = overlay.shape[:2]
        px, py = int(px), int(py)
        x1, y1 = max(0, px), max(0, py)
        x2, y2 = min(wc, px+ow), min(hc, py+oh)
        if x2 <= x1 or y2 <= y1:
            return
        ox, oy = x1-px, y1-py
        hs, ws = y2-y1, x2-x1
        a = alpha[oy:oy+hs, ox:ox+ws].astype(np.float32) / 255.0
        a3 = a[:, :, np.newaxis]
        fg = overlay[oy:oy+hs, ox:ox+ws, :3].astype(np.float32)
        bg = img[y1:y2, x1:x2].astype(np.float32)
        img[y1:y2, x1:x2] = (fg*a3 + bg*(1-a3)).astype(np.uint8)

    def get_tunnel_void(self, w, h, ocx, ocy, cw, ch):
        """Boyut bazlı cached tunnel void."""
        vw, vh = w-4, h-4
        if vw <= 0 or vh <= 0:
            return None
        
        # Işık yönü her seferinde farklı olsun diye cache kullanmayalım
        # ama en azından np hesaplarını basitleştirelim
        vx, vy = cw//2 - ocx, ch//2 - ocy
        lcx = np.clip(vw//2 + int(vx*0.3), 0, vw)
        lcy = np.clip(vh//2 + int(vy*0.3), 0, vh)
        
        # Boyut bazlı distance map cache
        key = (vw, vh)
        if key not in self._void_cache:
            Y, X = np.ogrid[:vh, :vw]
            # center (vw//2, vh//2) için base distance map
            base_dm = np.sqrt(((X - vw//2)**2 + (Y - vh//2)**2).astype(np.float32))
            md = np.sqrt(float(vw**2 + vh**2))
            self._void_cache[key] = (base_dm, md, Y, X)
        
        _, md, Y, X = self._void_cache[key]
        # Işık merkezine göre distance map (her seferinde farklı)
        dm = np.sqrt(((X - lcx)**2 + (Y - lcy)**2).astype(np.float32))
        nd = np.clip(dm / (md * 0.6), 0, 1)
        alpha = (np.power(nd, 0.7) * random.uniform(0.80, 0.98) * 255).astype(np.uint8)
        base_color = np.zeros((vh, vw, 3), dtype=np.uint8)
        return np.dstack([base_color, alpha])

    def get_random_background(self, w, h):
        if self.bg_images and random.random() < 0.9:
            bg = cv2.imread(random.choice(self.bg_images))
            if bg is not None:
                bg = cv2.resize(bg, (w, h))
                return (bg.astype(np.float32) * random.uniform(0.5, 0.95)).astype(np.uint8)
        val = random.randint(50, 110)
        bg = np.full((h, w, 3), val, dtype=np.uint8)
        noise = np.random.randint(-25, 25, bg.shape, dtype=np.int16)
        return np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    def generate_pallet(self):
        cls_id = CLASS_MAP['9859']
        
        pattern = random.choices(["ALTERNATING", "SANDWICH", "UNIFORM"],
                                 weights=[0.70, 0.20, 0.10], k=1)[0]
        num_rows = random.randint(8, 18)
        pkgs_per_row = random.randint(3, 6)
        pile_width = pkgs_per_row * self.pkg_w

        # Canvas boyutunu makul tut
        canvas_w = pile_width + random.randint(200, 400)
        canvas_h = (num_rows + 1) * max(self.pkg_h, self.rot_h) + 200
        # Minimum 1280
        canvas_w = max(canvas_w, self.width)
        canvas_h = max(canvas_h, self.height)

        wall = self.get_random_background(canvas_w, canvas_h)
        cur_y = canvas_h - random.randint(80, 150)
        start_x = (canvas_w - pile_width) // 2

        depth_cmds = []
        face_cmds = []

        for row_idx in range(num_rows):
            rtype = "HORIZONTAL"
            flip = False
            if pattern == "ALTERNATING" and row_idx % 2 == 1:
                flip = True
            elif pattern == "SANDWICH" and (num_rows//3 <= row_idx < 2*(num_rows//3)):
                rtype = "VERTICAL"

            if rtype == "HORIZONTAL":
                cur_y -= self.pkg_h
                if cur_y < 0:
                    break
                stamp = self.pkg_flipped if flip else self.pkg_normal
                stamp_alpha = stamp[:, :, 3]
                sh, sw = stamp.shape[:2]
                for col in range(pkgs_per_row):
                    x = start_x + col * self.pkg_w
                    self._add_commands(depth_cmds, face_cmds, stamp, stamp_alpha,
                                       x, cur_y, sw, sh, canvas_w, canvas_h)
            else:
                stamp = self.pkg_rotated
                stamp_alpha = stamp[:, :, 3]
                rh, rw = stamp.shape[:2]
                cur_y -= rh
                if cur_y < 0:
                    break
                v_pkgs = pile_width // rw if rw > 0 else 1
                off_x = start_x + (pile_width - v_pkgs * rw) // 2
                for col in range(v_pkgs):
                    x = off_x + col * rw
                    self._add_commands(depth_cmds, face_cmds, stamp, stamp_alpha,
                                       x, cur_y, rw, rh, canvas_w, canvas_h)

        # RENDER: Depth (reduced steps)
        for c in depth_cmds:
            if c['d'] <= 0:
                continue
            nx_v, ny_v = c['vx']/c['d'], c['vy']/c['d']
            tox = int(nx_v * c['dl'])
            toy = int(ny_v * c['dl'])
            body = np.full((c['sh'], c['sw'], 3), random.randint(30, 60), dtype=np.uint8)
            steps = min(int(max(abs(tox), abs(toy))), 12)
            for s in range(steps, 0, -2):
                r = s / steps if steps > 0 else 0
                self.overlay_fast(wall, body, c['x']+int(tox*r), c['y']+int(toy*r), c['alpha'])
            # Void
            if c['void'] is not None:
                self.overlay_fast(wall, c['void'], c['x']+2, c['y']+2, c['void'][:,:,3])

        # RENDER: Face
        for c in face_cmds:
            self.overlay_fast(wall, c['sprite'], c['x'], c['y'], c['sprite'][:,:,3])

        # Crop & resize to 1280x1280
        crop_top = max(0, cur_y - 100)
        cropped = wall[crop_top:canvas_h, :]
        final = cv2.resize(cropped, (self.width, self.height))

        # Annotations
        anns = []
        scale_x = self.width / cropped.shape[1]
        scale_y = self.height / cropped.shape[0]

        for c in face_cmds:
            sw_c, sh_c = c['sw'], c['sh']
            px_c = c['x']
            py_c = c['y'] - crop_top  # crop offset

            for i in range(PROFILES_PER_PKG):
                if sw_c > sh_c:
                    # Yatay paket
                    bw = sw_c / float(PROFILES_PER_PKG)
                    bh = float(sh_c)
                    bx = px_c + i * bw
                    by = float(py_c)
                else:
                    # Dikey paket (rotated)
                    bw = float(sw_c)
                    bh = sh_c / float(PROFILES_PER_PKG)
                    bx = float(px_c)
                    by = py_c + i * bh

                cx_n = ((bx + bw/2) * scale_x) / self.width
                cy_n = ((by + bh/2) * scale_y) / self.height
                nw = (bw * scale_x) / self.width
                nh = (bh * scale_y) / self.height

                if 0 <= cx_n <= 1 and 0 <= cy_n <= 1 and nw > 0 and nh > 0:
                    anns.append({'class_id': cls_id,
                                 'bbox': [cx_n, cy_n, nw, nh]})

        return final, anns

    def _add_commands(self, dl, fl, stamp, stamp_alpha, x, y, sw, sh, cw, ch):
        ocx, ocy = x + sw//2, y + sh//2
        vx, vy = cw//2 - ocx, ch//2 - ocy
        d = math.sqrt(vx**2 + vy**2)
        md = math.sqrt((cw/2)**2 + (ch/2)**2)
        dlen = min((d/md) * 60, sw * 0.8)
        void = self.get_tunnel_void(sw, sh, ocx, ocy, cw, ch)
        dl.append({
            'x': x, 'y': y, 'sw': sw, 'sh': sh,
            'alpha': stamp_alpha,
            'vx': vx, 'vy': vy, 'd': d, 'dl': dlen,
            'void': void
        })
        fl.append({
            'sprite': stamp, 'x': x, 'y': y,
            'sw': sw, 'sh': sh
        })


# ==========================================
# MAIN
# ==========================================
def main():
    if not os.path.exists(DXF_PROFILE):
        print(f"❌ {DXF_PROFILE} bulunamadı!")
        return

    print(f"\n🔧 9859 sprite hazırlanıyor (1 kez)...")
    
    # === 1 KEZ: DXF -> Sprite -> Paket ===
    base_sprite = build_base_sprite(DXF_PROFILE, RENDER_SIZE)
    if base_sprite is None:
        print("❌ Sprite üretilemedi!")
        return
    
    pkg_normal = build_package(base_sprite, PROFILES_PER_PKG)
    pkg_flipped = cv2.rotate(pkg_normal, cv2.ROTATE_180)
    pkg_rotated = cv2.rotate(pkg_normal, cv2.ROTATE_90_CLOCKWISE)
    
    print(f"   Sprite: {base_sprite.shape[1]}x{base_sprite.shape[0]}")
    print(f"   Paket:  {pkg_normal.shape[1]}x{pkg_normal.shape[0]}")
    print(f"   ✅ Hazır!")

    gen = PalletGenerator(pkg_normal, pkg_flipped, pkg_rotated, BG_FOLDER)

    for s in ['train', 'val']:
        os.makedirs(f'{BASE_DIR}/images/{s}', exist_ok=True)
        os.makedirs(f'{BASE_DIR}/labels/{s}', exist_ok=True)

    print(f"\n🚀 Script 2/3 TURBO: 9859 Başlıyor...")

    for i in range(IMAGES_PER_CLASS_TRAIN):
        if i % 20 == 0:
            print(f"   [Train] {i}/{IMAGES_PER_CLASS_TRAIN}")
        p, a = gen.generate_pallet()
        n = f"9859_train_{i}"
        cv2.imwrite(f'{BASE_DIR}/images/train/{n}.jpg', p)
        ls = "".join([f"{x['class_id']} {x['bbox'][0]:.6f} {x['bbox'][1]:.6f} {x['bbox'][2]:.6f} {x['bbox'][3]:.6f}\n" for x in a])
        with open(f'{BASE_DIR}/labels/train/{n}.txt', 'w') as f:
            f.write(ls)

    for i in range(IMAGES_PER_CLASS_VAL):
        if i % 5 == 0:
            print(f"   [Val] {i}/{IMAGES_PER_CLASS_VAL}")
        p, a = gen.generate_pallet()
        n = f"9859_val_{i}"
        cv2.imwrite(f'{BASE_DIR}/images/val/{n}.jpg', p)
        ls = "".join([f"{x['class_id']} {x['bbox'][0]:.6f} {x['bbox'][1]:.6f} {x['bbox'][2]:.6f} {x['bbox'][3]:.6f}\n" for x in a])
        with open(f'{BASE_DIR}/labels/val/{n}.txt', 'w') as f:
            f.write(ls)

    print(f"\n✅ Script 2/3 TAMAMLANDI!")


if __name__ == "__main__":
    main()

### data.yaml

In [ ]:
"""3 script bittikten sonra çalıştır - data.yaml oluşturur ve istatistik verir."""
import os

BASE_DIR = "dataset_5_profile_FINAL"
CLASS_ORDER = ['1524', '1560', '9859', '7170', '9794']

def main():
    with open(f'{BASE_DIR}/data.yaml', 'w') as f:
        names = "\n".join([f"  {i}: '{n}'" for i, n in enumerate(CLASS_ORDER)])
        f.write(f"path: {os.path.abspath(BASE_DIR)}\ntrain: images/train\nval: images/val\nnames:\n{names}\nnc: {len(CLASS_ORDER)}")
    print(f"\n📊 Dataset İstatistikleri:")
    for subset in ['train', 'val']:
        imgs = len([f for f in os.listdir(f'{BASE_DIR}/images/{subset}') if f.endswith('.jpg')])
        lbls = len([f for f in os.listdir(f'{BASE_DIR}/labels/{subset}') if f.endswith('.txt')])
        print(f"   {subset}: {imgs} resim, {lbls} etiket")
    for cls in CLASS_ORDER:
        t = len([f for f in os.listdir(f'{BASE_DIR}/images/train') if f.startswith(cls)])
        v = len([f for f in os.listdir(f'{BASE_DIR}/images/val') if f.startswith(cls)])
        print(f"   {cls}: Train={t}, Val={v}")
    print("✅ data.yaml oluşturuldu!")

if __name__ == "__main__":
    main()


In [ ]:
print(BASE_DIR)

In [ ]:
ls dataset_5_profile_FINAL

In [ ]:
cat dataset_5_profile_FINAL/data.yaml

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import os

# --- 1. DOSYA YOLU KONTROLÜ ---
# Kaggle'da veriyi ürettiğin klasörün adını buraya gir.
# Örnek: '/kaggle/working/dataset_5_profile_FINAL/data.yaml'
YAML_YOLU = '/kaggle/working/dataset_5_profile_FINAL/data.yaml' 

if not os.path.exists(YAML_YOLU):
    print(f"❌ HATA: {YAML_YOLU} bulunamadı! Yolun doğru olduğundan emin ol.")
else:
    print(f"✅ Yaml dosyası bulundu: {YAML_YOLU}")
    print("🚀 Model İndiriliyor ve Eğitim Başlıyor...")

    # --- 2. MODELİ BAŞLAT ---
    model = YOLO('yolo11n.pt')

    # --- 3. EĞİTİM ---
    results = model.train(
        data=YAML_YOLU,
        
        # Sonuçların kaydedileceği klasör
        project='/kaggle/working/runs/detect',
        name='yolo11n_1280_FINAL_Model',
        exist_ok=True,

        # --- HIZ & PERFORMANS (Kaggle 16GB VRAM T4/P100 Optimizasyonu) ---
        imgsz=1280,      
        epochs=150,      
        batch=4,        # ÇÖKMEMESİ İÇİN 16'DA KALMALI
        workers=2,       
        device=0,        

        # --- OPTİMİZASYON (Kendi Başarılı Ayarların) ---
        optimizer='auto', lr0=0.01, lrf=0.01, cos_lr=True, warmup_epochs=3, patience=30,
        
        # --- AUGMENTATION ---
        degrees=25.0, translate=0.1, scale=0.5, shear=2.0, perspective=0.0005,
        flipud=0.5, fliplr=0.5, mosaic=1.0, mixup=0.10, copy_paste=0.1,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

        # --- KAYIT ---
        plots=True, 
        save=True, 
        val=True, 
        save_period=1     
    )

    print("🎉 EĞİTİM TAMAMLANDI! '/kaggle/working/runs/detect/' klasöründen best.pt'yi indirebilirsin.")